# Adverse Competition in Concentrated Liquidity AMMs

## Evidence from Uniswap V3

### Research Question

In concentrated liquidity AMMs, large liquidity providers (LPs) can concentrate capital around the active tick, capturing a disproportionate share of fee revenue. We term this **adverse competition** — an LP-vs-LP dynamic distinct from **adverse selection** (LVR), which is a trader-vs-LP dynamic.

We construct a **congestion index** $\Delta I_t$ from a structural state-space model that captures LP repositioning unexplained by market conditions. We then test whether $\Delta I_t$ has a negative, statistically significant impact on fee-adjusted returns, **orthogonal to LVR**.

### Two-Stage Estimation

**Stage 1 — Extract congestion index:**

$$\frac{\Delta L_t}{L_{t-1}} = \beta_1 \frac{\Delta P_t}{P_{t-1}} + \beta_2 \cdot \text{txActivity}_t + e_t, \quad e_t = \gamma e_{t-1} + v_t$$

$$\Delta I_t \equiv e_t$$

**Stage 2 — Test adverse competition impact (orthogonal to LVR):**

$$\Delta \text{feeYield}_t = \alpha + \delta_1 \left|\frac{\Delta P_t}{P_{t-1}}\right| + \eta_t \quad \text{(strip LVR)}$$

$$\eta_t = \mu + \delta_2 \cdot \Delta I_t + \varepsilon_t \quad \text{(HC1 robust SEs)}$$

**Key result:** $\delta_2 = -0.002$, $z = -3.74$, $p < 0.001$ on 1,731 daily observations.

### Connection to Product Design

$\Delta I_t$ is the state variable that drives the **congestionToken** sigmoid pricing function $p(I) = \sigma(I/\lambda)$. The statistical significance of $\delta_2$ validates the economic mechanism — congestion creates measurable fee compression — that generates hedging demand for the instrument.

## 1. Data

**Pool:** Uniswap V3 USDC/WETH (`0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640`), 5 bps fee tier.

**Sample:** 1,760 daily observations (pool lifetime), ~11M transactions.

**Variables:**
| Variable | Definition |
|---|---|
| $\text{tvlUSD}_t$ | Total value locked (USD) |
| $\text{volumeUSD}_t$ | Daily trading volume (USD) |
| $\text{feesUSD}_t$ | Daily fee revenue (USD) |
| $P_t$ | token0 price (USDC per WETH) |
| $\text{txCount}_t$ | Daily transaction count |

**Derived series:**
| Series | Construction |
|---|---|
| $\Delta L_t / L_{t-1}$ | `delta(tvlUSD) / lagged(tvlUSD)` |
| $\Delta P_t / P_{t-1}$ | `delta(priceUSD) / lagged(priceUSD)` |
| $\text{txActivity}_t$ | `txCount / rolling_mean(txCount, 30)` |
| $\text{feeYield}_t$ | `feesUSD / tvlUSD` |
| $\lvert\Delta P / P\rvert$ | Absolute price returns (LVR proxy) |

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from data.DataHandler import (
    PoolEntryData, delta, tvlUSD, priceUSD, volumeUSD, feesUSD,
    div, lagged, txCount, normalize
)
from data.Econometrics import (
    LiquidityStateModel, AdverseCompetitionModel,
    beta, rho, state, result, delta_coeff, residual, ols_result
)
from data.UniswapClient import UniswapClient, v3

# ── Plotly monochrome template ──────────────────────────────────
classic = go.layout.Template()
classic.layout = go.Layout(
    font=dict(family="Courier New, monospace", size=12, color="#1a1a1a"),
    paper_bgcolor="#fafaf5",
    plot_bgcolor="#fafaf5",
    title=dict(font=dict(size=16, family="Courier New, monospace")),
    xaxis=dict(
        showgrid=True, gridcolor="#cccccc", gridwidth=0.5,
        linecolor="#1a1a1a", linewidth=1, mirror=True,
        zeroline=True, zerolinecolor="#999999", zerolinewidth=0.8
    ),
    yaxis=dict(
        showgrid=True, gridcolor="#cccccc", gridwidth=0.5,
        linecolor="#1a1a1a", linewidth=1, mirror=True,
        zeroline=True, zerolinecolor="#999999", zerolinewidth=0.8
    ),
    colorway=["#1a1a1a", "#666666", "#999999", "#bbbbbb"],
)
pio.templates["classic"] = classic
pio.templates.default = "classic"

# ── Load data ───────────────────────────────────────────────────
V3_USDC_WETH = "0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640"
client = UniswapClient(v3())
pool = PoolEntryData(V3_USDC_WETH, client=client)
pool_data = pool(pool.lifetimeLen())

# ── Summary statistics ──────────────────────────────────────────
summary = pool_data[["tvlUSD", "volumeUSD", "feesUSD", "token0Price", "txCount"]].describe()
summary.columns = ["TVL (USD)", "Volume (USD)", "Fees (USD)", "Price (USDC/WETH)", "Tx Count"]
print(f"Pool: V3 USDC/WETH 5bps")
print(f"Observations: {len(pool_data)}")
print(f"Period: {pool_data.index[0].date()} to {pool_data.index[-1].date()}")
print()
summary.round(2)

## 2. Stage 1 — Congestion Index Extraction

### Model Specification

We decompose daily liquidity changes into a market-explained component and a structural residual:

$$\frac{\Delta L_t}{L_{t-1}} = \underbrace{\beta_1 \frac{\Delta P_t}{P_{t-1}} + \beta_2 \cdot \text{txActivity}_t}_{\text{market state } A_t} + e_t$$

$$e_t = \gamma e_{t-1} + v_t, \quad v_t \sim \text{WN}(0, \sigma_v^2)$$

The **congestion index** $\Delta I_t \equiv e_t$ captures liquidity changes unexplained by market conditions — the structural residual from an unobserved components model with AR(1) dynamics.

### Hypotheses

| Parameter | Hypothesis | Economic Meaning |
|---|---|---|
| $\gamma > 0$ | Persistence | LP repositioning creates lasting effects — congestion carries over |
| $\gamma < 1$ | Stationarity | Congestion mean-reverts — sigmoid pricing $p(I)$ stays bounded |
| $\beta_1, \beta_2$ significant | Market state matters | Price and activity drive expected liquidity changes |

### Estimation

Unobserved components model (`statsmodels.UnobservedComponents`) with:
- Endogenous: $\Delta L_t / L_{t-1}$
- Exogenous: $[\Delta P_t / P_{t-1}, \; \text{txActivity}_t]$
- AR order: 1
- Internal z-score standardization (resolves 10⁹ scale ratio between endog and exog)

In [ ]:
# ── Stage 1: Congestion Index ΔI_t ─────────────────────────────
endog = div(delta(tvlUSD(pool_data)), lagged(tvlUSD(pool_data)))
exog = pd.DataFrame({
    "delta_price": div(delta(priceUSD(pool_data)), lagged(priceUSD(pool_data))),
    "tx_activity": normalize(txCount(pool_data), window=30),
})

ls = LiquidityStateModel()(endog=endog, exog=exog)

print("Stage 1: Congestion Index Extraction")
print("=" * 50)
print(f"γ (AR persistence) = {rho(ls):.4f}")
print(f"  0 < γ < 1: {0 < rho(ls) < 1}  (persistent + stationary)")
print()

res = result(ls)
print("Market state coefficients:")
for k, v in beta(ls).items():
    pval = res.pvalues[k]
    print(f"  {k}: β = {v:.6f}, p = {pval:.2e} {'***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''}")
print()
print(f"Observations: {len(state(ls))}")
print(f"ΔI_t mean: {state(ls).mean():.4f}")
print(f"ΔI_t std:  {state(ls).std():.4f}")

In [ ]:
# ── Stage 1: Time series panels ────────────────────────────────
# Compute market state prediction for plotting
mask = np.isfinite(endog) & exog.apply(np.isfinite).all(axis=1)
market_state = pd.Series(np.nan, index=endog.index)
clean_idx = endog[mask].index
fitted = result(ls).fittedvalues
if len(fitted) == len(clean_idx):
    market_state.loc[clean_idx] = fitted.values

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    subplot_titles=(
        "ΔL_t / L_{t-1}  (liquidity changes)",
        "E[ΔL_t | A_t]  (market state prediction)",
        f"ΔI_t  (congestion index,  γ = {rho(ls):.4f})"
    )
)

fig.add_trace(go.Scatter(
    x=endog.index, y=endog.values, mode="lines",
    line=dict(color="#1a1a1a", width=1), showlegend=False
), row=1, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="#999999", line_width=0.5, row=1, col=1)

fig.add_trace(go.Scatter(
    x=market_state.index, y=market_state.values, mode="lines",
    line=dict(color="#666666", width=1), showlegend=False
), row=2, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="#999999", line_width=0.5, row=2, col=1)

fig.add_trace(go.Scatter(
    x=state(ls).index, y=state(ls).values, mode="lines",
    line=dict(color="#1a1a1a", width=1), showlegend=False
), row=3, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="#999999", line_width=0.5, row=3, col=1)

fig.update_layout(height=750, margin=dict(t=40, b=30))
fig.show()

### Interpretation

**Persistence:** $\gamma = 0.78$ — the congestion index is persistent ($\gamma > 0$) but stationary ($\gamma < 1$). LP repositioning creates lasting effects on liquidity distribution, but these effects mean-revert. This is the structural parameter that justifies the sigmoid pricing function $p(I) = \sigma(I/\lambda)$ remaining bounded.

**Market state:** Both $\beta_1$ (price changes) and $\beta_2$ (transaction activity) are significant at $p < 0.001$. The market state $A_t$ captures expected liquidity changes driven by price movement and on-chain activity. What remains — $\Delta I_t$ — is the unexplained component attributable to strategic LP positioning.

**Connection to product design:** $\Delta I_t$ is the state variable for the congestionToken. The payoff notes specify the bound $|\Delta I_t| \leq \kappa \Delta L_t$ where $\kappa$ is the maximum observed shock over a rolling window. The stationarity of $\Delta I_t$ ($\gamma < 1$) ensures the integrated payoff $\varphi(I) = \lambda \cdot \ln(1 + e^{I/\lambda})$ has controlled growth.